# ComicVideoAI — API Mode

Notebook này khởi động ComicVideoAI ở chế độ **headless API** thay vì UI.
Phù hợp cho bot Telegram, automation, và orchestration từ xa.

Sau khi chạy, bạn sẽ có 1 public URL để gọi API tạo video.

In [ ]:
#@title 1) Cấu hình
GITHUB_REPO = "https://github.com/tqtuan8788-ai/comic-video.git"
BRANCH = "main"
REPO_DIR = "/content/comic-video"
PROJECT_DIR = f"{REPO_DIR}/colab_package"

# API Keys (để trống nếu không dùng)
DEEPSEEK_API_KEY = ""  #@param {type:"string"}
GEMINI_API_KEY = ""  #@param {type:"string"}

# Ports
API_PORT = 8000
EDGE_TTS_PORT = 5050
OMNIVOICE_PORT = 7861

# OmniVoice (tốn thời gian tải model, chỉ bật nếu cần voice cloning)
START_OMNIVOICE = False  #@param {type:"boolean"}

print(f"Project dir: {PROJECT_DIR}")
print(f"API port: {API_PORT}")
print(f"OmniVoice: {'ON' if START_OMNIVOICE else 'OFF'}")

In [ ]:
#@title 2) Cài system packages
import os, subprocess

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

run("apt-get update -qq && apt-get install -y -qq ffmpeg")
run("ffmpeg -version | head -1")
run("node --version && npm --version")

In [ ]:
#@title 3) Clone hoặc cập nhật code
import os, subprocess

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

if not os.path.exists(REPO_DIR):
    run(f"git clone --branch {BRANCH} --depth 1 {GITHUB_REPO} {REPO_DIR}")
else:
    run(f"git fetch --depth 1 origin {BRANCH}", cwd=REPO_DIR)
    run(f"git checkout {BRANCH}", cwd=REPO_DIR)
    run(f"git pull --ff-only origin {BRANCH}", cwd=REPO_DIR)

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

In [ ]:
#@title 4) Cài Node.js dependencies
import os, subprocess

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

os.chdir(PROJECT_DIR)
run("npm ci --no-audit --no-fund")

In [ ]:
#@title 5) Cấu hình .env.local
import os
from pathlib import Path

os.chdir(PROJECT_DIR)

env = {
    "PROVIDER_PRIORITY": "deepseek,gemini,pollinations,tts_free",
    "DEEPSEEK_API_KEY": DEEPSEEK_API_KEY,
    "DEEPSEEK_BASE_URL": "https://api.deepseek.com",
    "DEEPSEEK_MODEL_TEXT": "deepseek-chat",
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "POLLINATIONS_IMAGE_URL": "https://image.pollinations.ai/prompt",
    "TTS_FREE_PROVIDER": "edge_tts",
    "TTS_FREE_URL": "/api/edge-tts/tts",
    "EDGE_TTS_PROXY_TARGET": f"http://127.0.0.1:{EDGE_TTS_PORT}",
    "OMNIVOICE_PROXY_TARGET": f"http://127.0.0.1:{OMNIVOICE_PORT}",
    "USE_FREE_ONLY": "false",
}

Path(".env.local").write_text(
    "\n".join(f"{k}={v}" for k, v in env.items()) + "\n",
    encoding="utf-8"
)
print(".env.local configured")
for k, v in env.items():
    masked = v[:20] + "..." if len(v) > 20 else v
    print(f"  {k}={masked}")

In [ ]:
#@title 6) Khởi động tất cả services
import os, subprocess, time, pathlib

def run(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, cwd=cwd, shell=True, check=check)

os.chdir(PROJECT_DIR)

# ── Start Edge TTS ──
print("--- Starting Edge TTS ---")
edge_log = open("edge-tts.log", "w")
edge_env = os.environ.copy()
edge_env.update({"EDGE_TTS_PORT": str(EDGE_TTS_PORT)})
edge = subprocess.Popen(
    ["python3", "scripts/openai_edge_tts_server.py"],
    env=edge_env, stdout=edge_log, stderr=subprocess.STDOUT
)
pathlib.Path("edge-tts.pid").write_text(str(edge.pid))
print(f"Edge TTS PID: {edge.pid}")
time.sleep(3)
run(f"curl -sS http://127.0.0.1:{EDGE_TTS_PORT}/health || echo 'Waiting...'", check=False)

# ── Start OmniVoice (optional) ──
if START_OMNIVOICE:
    print("\n--- Starting OmniVoice ---")
    run("pip -q install omnivoice flask flask-cors soundfile")
    omni_log = open("omnivoice.log", "w")
    omni_env = os.environ.copy()
    omni_env.update({"OMNIVOICE_PORT": str(OMNIVOICE_PORT)})
    omni = subprocess.Popen(
        ["python3", "scripts/omnivoice-api-server.py"],
        env=omni_env, stdout=omni_log, stderr=subprocess.STDOUT
    )
    pathlib.Path("omnivoice.pid").write_text(str(omni.pid))
    print(f"OmniVoice PID: {omni.pid} (model loading...)")
else:
    print("\n--- OmniVoice: SKIPPED ---")

# ── Start API Server ──
print("\n--- Starting API Server ---")
api_env = os.environ.copy()
api_env.update({
    "API_PORT": str(API_PORT),
    "EDGE_TTS_PROXY_TARGET": f"http://127.0.0.1:{EDGE_TTS_PORT}",
    "OMNIVOICE_PROXY_TARGET": f"http://127.0.0.1:{OMNIVOICE_PORT}",
    "DEEPSEEK_API_KEY": DEEPSEEK_API_KEY,
    "GEMINI_API_KEY": GEMINI_API_KEY,
})
api_log = open("api-server.log", "w")
api = subprocess.Popen(
    ["npx", "tsx", "scripts/api-server.ts"],
    env=api_env, stdout=api_log, stderr=subprocess.STDOUT
)
pathlib.Path("api-server.pid").write_text(str(api.pid))
print(f"API Server PID: {api.pid}")
time.sleep(5)

# Health check
result = run(f"curl -sS http://127.0.0.1:{API_PORT}/health", check=False)
print(f"\nAPI Health: {result.returncode == 0 and 'OK' or 'FAILED'}")

In [ ]:
#@title 7) Mở public URL bằng ngrok
NGROK_TOKEN = ""  #@param {type:"string"}

import os, subprocess, time, re, pathlib

os.chdir(PROJECT_DIR)

if not NGROK_TOKEN:
    print("⚠️ Không có NGROK_TOKEN. API chỉ chạy local trên port", API_PORT)
    print(f"   Test: !curl http://127.0.0.1:{API_PORT}/health")
else:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok
    
    ngrok.set_auth_token(NGROK_TOKEN)
    
    # Dọn tunnel cũ
    try:
        for t in ngrok.get_tunnels():
            print(f"Closing old tunnel: {t.public_url}")
            ngrok.disconnect(t.public_url)
    except Exception as e:
        print(f"Cleanup note: {e}")
    
    # Mở tunnel tới API port
    public_url = ngrok.connect(API_PORT, bind_tls=True).public_url
    print(f"\n✅ API Public URL: {public_url}")
    print(f"   Health:  {public_url}/health")
    print(f"   Jobs:    {public_url}/jobs")
    print(f"\n   Test: curl {public_url}/health")
    print(f"   Create: curl -X POST {public_url}/jobs \\")
    print(f"     -H 'Content-Type: application/json' \\")
    print(f"     -d '{{\"script\":\"Cậu bé tìm thấy chiếc đồng hồ biết nói...\",\"artStyle\":\"comic_manhua\",\"voiceName\":\"vi-VN-HoaiMyNeural\"}}'")

In [ ]:
#@title 8) Quick test — tạo 1 video ngắn (2 scenes)
import requests, json, time

API_URL = f"http://127.0.0.1:{API_PORT}"

# 1) Create job
print("Creating job...")
resp = requests.post(f"{API_URL}/jobs", json={
    "script": "Cậu bé mở chiếc hộp bí ẩn và phát hiện ra một thế giới thu nhỏ bên trong.",
    "artStyle": "comic_manhua",
    "voiceName": "vi-VN-HoaiMyNeural",
    "sceneCount": 2
})
job = resp.json()
print(json.dumps(job, indent=2, ensure_ascii=False))
job_id = job["id"]

# 2) Poll status
print(f"\nPolling job {job_id}...")
for _ in range(30):
    time.sleep(10)
    resp = requests.get(f"{API_URL}/jobs/{job_id}")
    status = resp.json()
    print(f"  [{status['status']}] {status['progress']}: {status.get('progressDetail', '')}")
    if status["status"] in ("done", "failed"):
        break

# 3) Get result
if status["status"] == "done":
    print("\n✅ Done!")
    result = requests.get(f"{API_URL}/jobs/{job_id}/result").json()
    print(f"Scenes: {result['sceneCount']}")
    print(f"Duration: {result['totalDuration']}s")
    for asset in result.get("assets", []):
        print(f"  Scene {asset['index']+1}: image={asset['hasImage']}, audio={asset['hasAudio']}")
else:
    print(f"\n❌ Failed: {status}")

In [ ]:
#@title Dừng tất cả services
import pathlib, subprocess, os

os.chdir(PROJECT_DIR)
for pid_file in ["edge-tts.pid", "omnivoice.pid", "api-server.pid"]:
    p = pathlib.Path(pid_file)
    if p.exists():
        pid = p.read_text().strip()
        if pid:
            subprocess.run(f"kill {pid} 2>/dev/null || true", shell=True)
            print(f"Stopped {pid_file} (PID {pid})")

print("All services stopped.")